# Notebook 05 – Final Evaluation & Comparison

- Master comparison table (all models, RMSE / MAE)
- Top-K metric comparison
- Error analysis by user activity
- Error analysis by item popularity
- Final summary plots


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation           import rating_metrics, rmse
from src.baselines            import BiasModel
from src.knn_cf               import SurpriseKNN
from src.matrix_factorization import SurpriseMF

sns.set_style('whitegrid')
os.makedirs('../results', exist_ok=True)
SEED = 42
np.random.seed(SEED)

In [ ]:
train  = pd.read_parquet('../data/train.parquet')
test   = pd.read_parquet('../data/test.parquet')
movies = pd.read_csv('../data/movie.csv')
print(f'Train: {len(train):,}   Test: {len(test):,}')

## 5.1 Master comparison table

In [ ]:
# Load results saved by run_all.py
df_all = pd.read_csv('../results/all_model_results.csv', index_col=0)
print('=== All Models – Rating Prediction Metrics ===')
print(df_all.to_string())

## 5.2 Summary bar chart

In [ ]:
n_models = len(df_all)
colors   = (['#4C72B0'] * 4 + ['#DD8452', '#55A868'])[:n_models]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_all['RMSE'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Test RMSE – All Models', fontsize=13)
axes[0].set_ylabel('RMSE (lower is better)')
axes[0].set_xticklabels(df_all.index, rotation=35, ha='right')
for i, v in enumerate(df_all['RMSE']):
    axes[0].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=8)

df_all['MAE'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Test MAE – All Models', fontsize=13)
axes[1].set_ylabel('MAE (lower is better)')
axes[1].set_xticklabels(df_all.index, rotation=35, ha='right')
for i, v in enumerate(df_all['MAE']):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=8)

plt.suptitle('Movie Recommendation – Model Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Top-K metrics comparison

In [ ]:
knn_topk = pd.read_csv('../results/knn_topk_metrics.csv')
mf_topk  = pd.read_csv('../results/mf_topk_metrics.csv')

topk_df = pd.concat([
    knn_topk.assign(Model='KNN'),
    mf_topk.assign(Model='MF')
]).set_index('Model')

print('=== Top-10 Ranking Metrics ===')
print(topk_df.to_string())

topk_df.plot(kind='bar', figsize=(10, 4), edgecolor='white')
plt.title('Top-10 Ranking Metrics – KNN vs MF')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/topk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Error analysis – by user activity bucket

In [ ]:
n_users  = int(train['user_idx'].max()) + 1
n_movies = int(train['movie_idx'].max()) + 1

# Activity buckets
user_activity = train.groupby('userId').size().rename('n_train_ratings')
test_activity = test.merge(user_activity.reset_index(), on='userId', how='left')

bins   = [0, 10, 50, 200, np.inf]
labels = ['<=10', '11-50', '51-200', '>200']
test_activity['bucket'] = pd.cut(
    test_activity['n_train_ratings'], bins=bins, labels=labels
)
print('Test samples per activity bucket:')
print(test_activity['bucket'].value_counts().sort_index())

In [ ]:
# Re-fit best models
bm = BiasModel(n_users=n_users, n_items=n_movies)
bm.fit(train)
test_activity['bias_pred'] = bm.predict(test_activity)

knn = SurpriseKNN(k=40)
knn.fit(train)
test_activity['knn_pred'] = knn.predict_df(test_activity)

# Load best d and lambda from saved results
df_d   = pd.read_csv('../results/mf_d_sensitivity.csv',      index_col='d')
df_lam = pd.read_csv('../results/mf_lambda_sensitivity.csv', index_col='lambda')
best_d   = int(df_d['RMSE'].idxmin())
best_lam = float(df_lam['RMSE'].idxmin())

mf = SurpriseMF(d=best_d, n_epochs=20, reg_all=best_lam)
mf.fit(train)
test_activity['mf_pred'] = mf.predict_df(test_activity)

In [ ]:
bucket_rows = []
for bucket, grp in test_activity.groupby('bucket', observed=True):
    true = grp['rating'].values
    bucket_rows.append({
        'Bucket':          str(bucket),
        'N':               len(grp),
        'BiasModel RMSE':  rmse(true, grp['bias_pred'].values),
        'KNN RMSE':        rmse(true, grp['knn_pred'].values),
        'MF RMSE':         rmse(true, grp['mf_pred'].values),
    })

df_bucket = pd.DataFrame(bucket_rows).set_index('Bucket')
print(df_bucket.to_string())
df_bucket.to_csv('../results/error_analysis_by_activity.csv')

df_bucket[['BiasModel RMSE', 'KNN RMSE', 'MF RMSE']].plot(
    kind='bar', figsize=(9, 5), edgecolor='white'
)
plt.title('RMSE by User Activity Bucket')
plt.xlabel('# training ratings per user')
plt.ylabel('RMSE')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../results/error_analysis_activity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5 Error analysis – by item popularity bucket

In [ ]:
item_popularity = train.groupby('movieId').size().rename('n_train_ratings_item')
test_pop = test.merge(item_popularity.reset_index(), on='movieId', how='left')
test_pop['n_train_ratings_item'] = test_pop['n_train_ratings_item'].fillna(0)

pbins   = [0, 10, 100, 1000, np.inf]
plabels = ['<=10', '11-100', '101-1000', '>1000']
test_pop['pop_bucket'] = pd.cut(
    test_pop['n_train_ratings_item'], bins=pbins, labels=plabels
)

test_pop['bias_pred'] = bm.predict(test_pop)
test_pop['knn_pred']  = knn.predict_df(test_pop)
test_pop['mf_pred']   = mf.predict_df(test_pop)

pop_rows = []
for bucket, grp in test_pop.groupby('pop_bucket', observed=True):
    true = grp['rating'].values
    pop_rows.append({
        'Bucket':          str(bucket),
        'N':               len(grp),
        'BiasModel RMSE':  rmse(true, grp['bias_pred'].values),
        'KNN RMSE':        rmse(true, grp['knn_pred'].values),
        'MF RMSE':         rmse(true, grp['mf_pred'].values),
    })

df_pop = pd.DataFrame(pop_rows).set_index('Bucket')
print(df_pop.to_string())
df_pop.to_csv('../results/error_analysis_by_popularity.csv')

df_pop[['BiasModel RMSE', 'KNN RMSE', 'MF RMSE']].plot(
    kind='bar', figsize=(9, 5), edgecolor='white'
)
plt.title('RMSE by Item Popularity Bucket')
plt.xlabel('# training ratings per movie')
plt.ylabel('RMSE')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../results/error_analysis_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.6 Final summary

In [ ]:
print('='*50)
print('FINAL RESULTS SUMMARY')
print('='*50)
print(df_all.to_string())
print('\nAll plots and CSVs saved to results/')
print('\nFiles generated:')
for f in sorted(os.listdir('../results')):
    if not f.startswith('.'):
        print(f'  results/{f}')